In [1]:
!pip -q install rdkit biopython scikit-learn tqdm matplotlib seaborn networkx

!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.5 MB/s eta 0:00:00
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 153.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 121.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.

In [6]:
import os
import shutil

print("1. Preparing directories...")
os.makedirs("/content/datasets", exist_ok=True)

# Clone MolTrans if missing; this notebook uses BIOSNAP and BindingDB only
if not os.path.exists("/content/datasets/MolTrans"):
    print("Cloning MolTrans repository...")
    os.system("git clone -q https://github.com/KexinHuang12345/MolTrans.git /content/datasets/MolTrans")

print("2. Keeping only BIOSNAP and BindingDB datasets...")
keep_datasets = {"BIOSNAP", "BindingDB"}
dataset_root = "/content/datasets/MolTrans/dataset"

if os.path.exists(dataset_root):
    for dataset_name in os.listdir(dataset_root):
        dataset_path = os.path.join(dataset_root, dataset_name)
        if os.path.isdir(dataset_path) and dataset_name not in keep_datasets:
            shutil.rmtree(dataset_path)

print()
print("=== Available Datasets ===")
os.system("ls -l /content/datasets/MolTrans/dataset/")

print("Setup complete. You can now run the training loop.")

1. Preparing directories...
2. Keeping only BIOSNAP and BindingDB datasets...

=== Available Datasets ===
Setup complete. You can now run the training loop.


In [7]:
import glob

paths = glob.glob("/content/datasets/**/*", recursive=True)

pdb = [p for p in paths if p.lower().endswith(".pdb")]
cif = [p for p in paths if p.lower().endswith(".cif") or p.lower().endswith(".mmcif")]

print("Total files:", len(paths))
print("PDB count:", len(pdb))
print("mmCIF count:", len(cif))

print("\nSample PDB paths:")
for p in pdb[:10]:
    print(" ", p)

print("\nSample mmCIF paths:")
for p in cif[:10]:
    print(" ", p)

Total files: 50
PDB count: 0
mmCIF count: 0

Sample PDB paths:

Sample mmCIF paths:


In [9]:
# ============================
# 1D-CNN (DeepDTA style) DTI Model
# - Replaces 3D ProteinEGNN with a 1D Sequence CNN
# - Uses EXACT paper splits (train.csv, val.csv, test.csv)
# - Integrated Training: Merges Train/Val across datasets, tests independently
# - Added pos_weight for class imbalance
# - Added Dynamic Thresholding for F1 optimization
# ============================

import os, math, random, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from IPython.display import display
from sklearn.metrics import (
    roc_auc_score, f1_score, matthews_corrcoef, balanced_accuracy_score,
    average_precision_score, recall_score, confusion_matrix
)

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Draw, rdDepictor
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

# --- Config ---
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-4
FP_BITS = 2048
EMB_DIM = 64
MAX_SEQ_LEN = 1000  # Max length for 1D protein sequences
POCKET_K = 64    # Number of tokens the 1D CNN will output (mimicking pockets)
MIXED_PRECISION = True
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Metrics ---
@torch.no_grad()
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    out = {}
    try:
        out["auc"] = float(roc_auc_score(y_true, y_prob))
        out["auprc"] = float(average_precision_score(y_true, y_prob))
    except:
        out["auc"] = float("nan")
        out["auprc"] = float("nan")
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    out["sensitivity"] = float(recall_score(y_true, y_pred, zero_division=0))
    if len(y_true) > 0:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        out["specificity"] = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    else:
        out["specificity"] = 0.0
    return out

# --- Encoders & Fusion ---
class DrugEncoderFP(nn.Module):
    def __init__(self, fp_bits=2048, out_dim=128, hidden=512, n_tokens=4):
        super().__init__()
        self.n_tokens = n_tokens
        self.mlp = nn.Sequential(
            nn.Linear(fp_bits, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, out_dim * n_tokens)
        )
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, fp):
        z = self.mlp(fp)
        B = z.shape[0]
        z = z.view(B, self.n_tokens, -1)
        return self.norm(z)

class ProteinCNN1D(nn.Module):
    def __init__(self, vocab_size=25, emb_dim=128, num_filters=128, out_tokens=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=4)
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters*2, kernel_size=8)
        self.conv3 = nn.Conv1d(in_channels=num_filters*2, out_channels=emb_dim, kernel_size=12)
        self.pool = nn.AdaptiveMaxPool1d(out_tokens)

    def forward(self, seq):
        x = self.embed(seq)
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        return x.permute(0, 2, 1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=128, heads=4, layers=2, dropout=0.1):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim*4,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.pocket_encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.out = nn.Sequential(
            nn.Linear(dim*2, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1)
        )

    def forward(self, drug_tokens, pocket_tokens, pocket_pad_mask=None, return_attention=False):
        pocket_tokens = self.pocket_encoder(pocket_tokens, src_key_padding_mask=pocket_pad_mask)
        attn_out, attn_weights = self.cross_attn(
            query=drug_tokens,
            key=pocket_tokens,
            value=pocket_tokens,
            key_padding_mask=pocket_pad_mask,
            need_weights=return_attention,
            average_attn_weights=False
        )
        drug_pool = drug_tokens.mean(dim=1)
        inter_pool = attn_out.mean(dim=1)
        pair_embedding = torch.cat([drug_pool, inter_pool], dim=-1)
        logits = self.out(pair_embedding).squeeze(-1)
        if return_attention:
            return {
                "logits": logits,
                "attn_weights": attn_weights,
                "drug_tokens": drug_tokens,
                "pocket_tokens": pocket_tokens,
                "attn_out": attn_out,
                "pair_embedding": pair_embedding
            }
        return logits

class DTI1DModel(nn.Module):
    def __init__(self, fp_bits=2048, dim=128, pocket_k=128):
        super().__init__()
        self.drug_enc = DrugEncoderFP(fp_bits=fp_bits, out_dim=dim, n_tokens=4)
        self.prot_enc = ProteinCNN1D(vocab_size=26, emb_dim=dim, num_filters=dim, out_tokens=pocket_k)
        self.fusion = CrossAttentionFusion(dim=dim)

    def forward(self, fp, prot_seq, return_attention=False):
        drug_tokens = self.drug_enc(fp)
        prot_tokens = self.prot_enc(prot_seq)
        B, K, _ = prot_tokens.shape
        mask = torch.zeros((B, K), dtype=torch.bool, device=prot_tokens.device)
        return self.fusion(
            drug_tokens,
            prot_tokens,
            pocket_pad_mask=mask,
            return_attention=return_attention
        )

# --- Data Processing ---
def morgan_fp(smiles: str, n_bits=2048, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.astype(np.float32)

AA_MAP = {a: i for i, a in enumerate("ACDEFGHIKLMNPQRSTVWY", 1)}

def tokenize_protein(seq, max_len=1000):
    tokens = [AA_MAP.get(a, 21) for a in seq[:max_len]]
    if len(tokens) < max_len:
        tokens += [0] * (max_len - len(tokens))
    return np.array(tokens, dtype=np.int64)

class SequenceDTIDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fp = morgan_fp(str(row["smiles"]), n_bits=FP_BITS)
        seq_tokens = tokenize_protein(str(row["sequence"]), max_len=MAX_SEQ_LEN)
        return {
            "fp": torch.tensor(fp, dtype=torch.float32),
            "seq": torch.tensor(seq_tokens, dtype=torch.long),
            "y": torch.tensor(float(row["label"]), dtype=torch.float32)
        }

def seq_collate_fn(batch):
    fp = torch.stack([b["fp"] for b in batch], dim=0)
    seq = torch.stack([b["seq"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    return fp, seq, y

# --- Train Loop ---
def run_epoch(model, loader, optimizer=None, scaler=None, pos_weight=None, threshold=0.5):
    train = optimizer is not None
    model.train(train)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    all_y, all_p = [], []
    total_loss = 0.0

    for fp, seq, y in tqdm(loader, leave=False):
        fp, seq, y = fp.to(device), seq.to(device), y.to(device)
        if train: optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(scaler is not None)):
            logits = model(fp, seq)
            loss = bce(logits, y)

        if train:
            if scaler:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item()) * fp.size(0)
        prob = torch.sigmoid(logits).detach().cpu().numpy()
        all_p.extend(prob.tolist())
        all_y.extend(y.detach().cpu().numpy().tolist())

    return total_loss / max(1, len(loader.dataset)), compute_metrics(all_y, all_p, threshold=threshold)

@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_y, all_p = [], []
    for fp, seq, y in tqdm(loader, leave=False):
        fp, seq = fp.to(device), seq.to(device)
        logits = model(fp, seq)
        prob = torch.sigmoid(logits).detach().cpu().numpy()
        all_p.extend(prob.tolist())
        all_y.extend(y.detach().numpy().tolist())
    return np.array(all_y), np.array(all_p)

def find_best_threshold(y_true, y_prob):
    best_thresh = 0.5
    best_f1 = 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh, best_f1

# --- Execution ---
DATA_DIRS = {
    "BIOSNAP": "/content/datasets/MolTrans/dataset/BIOSNAP/full_data",
    "BindingDB": "/content/datasets/MolTrans/dataset/BindingDB"
}

def load_paper_split(data_dir, split_name):
    path = os.path.join(data_dir, split_name)
    if not os.path.exists(path):
        return pd.DataFrame()

    df = pd.read_csv(path)
    smiles_col = next((c for c in df.columns if 'smiles' in str(c).lower()), df.columns[0])
    target_col = next((c for c in df.columns if 'sequence' in str(c).lower() or 'target' in str(c).lower()), df.columns[1])
    label_col = next((c for c in df.columns if 'label' in str(c).lower() or 'affinity' in str(c).lower() or str(c).lower() == 'y'), None)
    if label_col is None:
        remaining = [c for c in df.columns if c not in [smiles_col, target_col]]
        label_col = remaining[0] if remaining else df.columns[2]

    df = df[[smiles_col, target_col, label_col]].copy()
    df.columns = ["smiles", "sequence", "label"]
    df["label"] = pd.to_numeric(df["label"], errors='coerce')
    df = df.dropna(subset=["label"])


    return df

def get_cold_splits(train_df, val_df, test_df):
    known_drugs = set(train_df['smiles']).union(set(val_df['smiles']))
    known_targets = set(train_df['sequence']).union(set(val_df['sequence']))
    cold_drug_df = test_df[~test_df['smiles'].isin(known_drugs)].copy()
    cold_target_df = test_df[~test_df['sequence'].isin(known_targets)].copy()
    cold_binding_df = test_df[(~test_df['smiles'].isin(known_drugs)) & (~test_df['sequence'].isin(known_targets))].copy()
    return cold_drug_df, cold_target_df, cold_binding_df

datasets = {}
for name, ddir in DATA_DIRS.items():
    datasets[name] = {
        "train": load_paper_split(ddir, "train.csv"),
        "val": load_paper_split(ddir, "val.csv"),
        "test": load_paper_split(ddir, "test.csv")
    }

configurations = ["Integrated"]

for config in configurations:
    print(f"\n{'='*50}")
    print(f"STARTING TRAINING CONFIGURATION: {config.upper()}")
    print(f"{'='*50}")

    if config == "Integrated":
        train_df = pd.concat([datasets[n]["train"] for n in datasets], ignore_index=True)
        val_df = pd.concat([datasets[n]["val"] for n in datasets], ignore_index=True)
    else:
        train_df = datasets[config]["train"]
        val_df = datasets[config]["val"]

    print(f"Training Samples: {len(train_df)} | Validation Samples: {len(val_df)}")

    num_pos = train_df["label"].sum()
    num_neg = len(train_df) - num_pos
    pos_weight_val = num_neg / max(num_pos, 1)
    print(f"Calculated global pos_weight: {pos_weight_val:.2f}")
    pw_tensor = torch.tensor([pos_weight_val], device=device, dtype=torch.float32)

    train_loader = DataLoader(SequenceDTIDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=seq_collate_fn)
    val_loader = DataLoader(SequenceDTIDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=seq_collate_fn)

    model = DTI1DModel(fp_bits=FP_BITS, dim=EMB_DIM, pocket_k=POCKET_K).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scaler = torch.amp.GradScaler("cuda", enabled=(MIXED_PRECISION and device.type=="cuda"))

    best_auprc = -1
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_metrics = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler, pos_weight=pw_tensor)
        va_loss, va_metrics = run_epoch(model, val_loader)

        auprc = va_metrics.get("auprc", 0)
        if epoch % 5 == 0 or epoch == EPOCHS:
            print(f"Epoch {epoch:02d} | Tr Loss: {tr_loss:.4f} | Tr F1: {tr_metrics.get('f1',0):.4f} | Va Loss: {va_loss:.4f} | Va F1: {va_metrics.get('f1',0):.4f} | Va AUPRC: {auprc:.4f}")

        if auprc > best_auprc:
            best_auprc = auprc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"\n==== TESTING RESULTS ({config}) ====")

    if config == "Integrated":
        for name in datasets:
            print(f"\n--- {name} Test Set ---")
            # First, find best threshold based on its specific validation set
            val_loader_specific = DataLoader(SequenceDTIDataset(datasets[name]["val"]), batch_size=BATCH_SIZE, shuffle=False, collate_fn=seq_collate_fn)
            y_val_true, y_val_prob = get_predictions(model, val_loader_specific)
            best_t, best_val_f1 = find_best_threshold(y_val_true, y_val_prob)
            print(f"Optimized Decision Threshold for {name}: {best_t:.2f} (Val F1: {best_val_f1:.4f})")

            te_loader = DataLoader(SequenceDTIDataset(datasets[name]["test"]), batch_size=BATCH_SIZE, shuffle=False, collate_fn=seq_collate_fn)
            _, te_metrics = run_epoch(model, te_loader, threshold=best_t)
            for k, v in te_metrics.items():
                print(f"{k.upper()}: {v:.4f}")
    else:
        te_df = datasets[config]["test"]
        y_val_true, y_val_prob = get_predictions(model, val_loader)
        best_t, best_val_f1 = find_best_threshold(y_val_true, y_val_prob)
        print(f"Optimized Decision Threshold: {best_t:.2f} (Val F1: {best_val_f1:.4f})")

        te_loader = DataLoader(SequenceDTIDataset(te_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=seq_collate_fn)
        _, te_metrics = run_epoch(model, te_loader, threshold=best_t)
        print(f"\n--- {config} Standard Test Set ---")
        for k, v in te_metrics.items():
            print(f"{k.upper()}: {v:.4f}")


Device: cuda

STARTING TRAINING CONFIGURATION: INTEGRATED
Training Samples: 31906 | Validation Samples: 9392
Calculated global pos_weight: 0.99


Epoch 05 | Tr Loss: 0.2643 | Tr F1: 0.8875 | Va Loss: 0.4959 | Va F1: 0.6617 | Va AUPRC: 0.7585


Epoch 10 | Tr Loss: 0.1464 | Tr F1: 0.9376 | Va Loss: 0.5862 | Va F1: 0.6805 | Va AUPRC: 0.7586


Epoch 15 | Tr Loss: 0.1049 | Tr F1: 0.9554 | Va Loss: 0.6564 | Va F1: 0.6918 | Va AUPRC: 0.7572


Epoch 20 | Tr Loss: 0.0826 | Tr F1: 0.9663 | Va Loss: 0.7763 | Va F1: 0.6887 | Va AUPRC: 0.7557


...

Epoch 50 | Tr Loss: 0.0483 | Tr F1: 0.9792 | Va Loss: 1.0016 | Va F1: 0.6959 | Va AUPRC: 0.7470

==== TESTING RESULTS (Integrated) ====

--- BIOSNAP Test Set ---


Optimized Decision Threshold for BIOSNAP: 0.45 (Val F1: 0.8269)


AUC: 0.8843
AUPRC: 0.8924
F1: 0.8109
SENSITIVITY: 0.8051
SPECIFICITY: 0.8166

--- BindingDB Test Set ---


Optimized Decision Threshold for BindingDB: 0.90 (Val F1: 0.6071)


AUC: 0.9091
AUPRC: 0.6193
F1: 0.6145
SENSITIVITY: 0.7213
SPECIFICITY: 0.8952
